Imports

In [1]:
import pandas as pd
from pathlib import Path
import re
import nltk
from nltk.tokenize import sent_tokenize
from transformers import pipeline
from tqdm.auto import tqdm
import torch

DEVICE = 0 if torch.cuda.is_available() else -1
#BATCH_SIZE_GO = 16
#BATCH_SIZE_ROBERTA = 32
BATCH_SIZE_VAD = 32

nltk.download('punkt_tab', quiet=True)

True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data_path = Path('/content/drive/MyDrive/User-Songs-Music-Group.xls')

if data_path.suffix.lower() in {'.xls', '.xlsx'}:
    try:
        analysis_df = pd.read_excel(data_path, engine='xlrd')
    except ImportError as exc:
        raise ImportError("xlrd is required to read .xls files. Run: !pip install xlrd") from exc
    except Exception:
        # Some files have .xls extension but are actually CSV text files.
        analysis_df = pd.read_csv(data_path)
else:
    analysis_df = pd.read_csv(data_path)

analysis_df.head()

,Unnamed: 0,created_at,music_source,title,artist,disorder,sentiment_direction,sentiment_score,music_url,lyric,emotion_anger_score,emotion_disgust_score,emotion_fear_score,emotion_joy_score,emotion_neutral_score,emotion_sadness_score,emotion_surprise_score,anonymized_author_id,anonymized_tweet_id
0,0,2021-06-25 01:18:42-04:00,spotify,Burnin Bridges / Long Day (feat. IDK),Quadeca,depression,POSITIVE,0.9971,https://open.spotify.com/track/0B5mjEPetRaVGBs...,Highest To Lowest: Quadeca LyricsQuadeca's Son...,0.0294,0.0014,0.0132,0.2346,0.0993,0.0832,0.5390,0ef1ff6f271d5cb3541f6995,e611b266fd0d9de7dd2696d2
1,1,2021-09-25 12:05:15-04:00,spotify,She's A Lady,Tom Jones,control,POSITIVE,0.9988,https://open.spotify.com/track/5QfQ5iROTerk3MZ...,She’s a Lady Lyrics[Verse 1]\nWell she's all y...,0.0271,0.0203,0.0071,0.1162,0.1262,0.0335,0.6696,12b35a1f4485cf4aff1a634a,c04e17b275379c024e6179a4
2,2,2022-05-14 16:57:16-04:00,spotify,Lilies of the Valley,David Byrne,control,POSITIVE,0.9532,https://open.spotify.com/track/5mwlD1lkAGHrd2l...,Lilies of the Valley Lyrics[Verse 1]\nMomma sh...,0.0252,0.0054,0.0556,0.5990,0.0678,0.2116,0.0354,12b35a1f4485cf4aff1a634a,a6b49eeb6e211e7940be6593
3,3,2021-05-24 10:04:29-04:00,spotify,School's Out,Alice Cooper,control,NEGATIVE,0.9995,https://open.spotify.com/track/5Z8EDau8uNcP1E8...,"School’s Out Lyrics[Verse 1]\nWell, we got no ...",0.1258,0.1330,0.0149,0.0306,0.1615,0.5014,0.0330,12b35a1f4485cf4aff1a634a,bc836b7694e9c922f374ee74
4,4,2022-01-09 15:12:28-05:00,spotify,Call My Friends,Shawn Mendes,depression,POSITIVE,0.9978,https://open.spotify.com/track/6KVxMOxlSYpC5nr...,"Call My Friends Lyrics[Verse 1]\nRight now, I'...",0.0016,0.0006,0.0015,0.0075,0.0055,0.9777,0.0057,3e06169cafdc66d26a9b9ebb,7970bb8c13ff94ee62102db0


# Pipeline for sentiment models

## In this part the VAD model will be used to get a better understanding of the lyrics

The model looks at the lyrics sentence by sentence and predicts Valence, Arousal, and Dominance scores.

Preprocessing

In [4]:
def preprocess_lyrics(text: str) -> str:
    if pd.isna(text):
        return ''
    
    text = str(text)
    text = text.replace('\r', '\n')
    text = re.sub(r'\n{2,}', '\n', text)
    text = text.strip()
    return text

In [5]:
def split_into_sentences(text: str) -> list:
    if not text or not text.strip():
        return []
    return [s.strip() for s in sent_tokenize(text) if s.strip()]

In [6]:
def build_text_classifier(model_name, top_k=None):
    return pipeline(
    'text-classification',
    model=model_name,
    tokenizer=model_name,
    top_k=top_k,
    truncation=True
    )

GoEmotions

In [8]:
GO_EMOTIONS_VA = {
    "admiration":      {"valence": 0.74, "arousal": 0.30},
    "amusement":       {"valence": 0.78, "arousal": 0.44},
    "anger":           {"valence": -0.64, "arousal": 0.60},
    "annoyance":       {"valence": -0.46, "arousal": 0.28},
    "approval":        {"valence": 0.60, "arousal": 0.20},
    "caring":          {"valence": 0.70, "arousal": 0.28},
    "confusion":       {"valence": -0.18, "arousal": 0.22},
    "curiosity":       {"valence": 0.32, "arousal": 0.46},
    "desire":          {"valence": 0.52, "arousal": 0.54},
    "disappointment":  {"valence": -0.52, "arousal": -0.22},
    "disapproval":     {"valence": -0.58, "arousal": 0.14},
    "disgust":         {"valence": -0.70, "arousal": 0.26},
    "embarrassment":   {"valence": -0.42, "arousal": 0.12},
    "excitement":      {"valence": 0.72, "arousal": 0.74},
    "fear":            {"valence": -0.64, "arousal": 0.60},
    "gratitude":       {"valence": 0.80, "arousal": 0.20},
    "grief":           {"valence": -0.78, "arousal": -0.38},
    "joy":             {"valence": 0.86, "arousal": 0.56},
    "love":            {"valence": 0.88, "arousal": 0.32},
    "nervousness":     {"valence": -0.40, "arousal": 0.52},
    "neutral":         {"valence": 0.00, "arousal": 0.00},
    "optimism":        {"valence": 0.72, "arousal": 0.36},
    "pride":           {"valence": 0.74, "arousal": 0.38},
    "realization":     {"valence": 0.14, "arousal": 0.22},
    "relief":          {"valence": 0.60, "arousal": -0.30},
    "remorse":         {"valence": -0.60, "arousal": -0.20},
    "sadness":         {"valence": -0.72, "arousal": -0.44},
    "surprise":        {"valence": 0.20, "arousal": 0.66},
}

In [9]:
go_classifier = pipeline(
    task='text-classification',
    model='SamLowe/roberta-base-go_emotions',
    top_k=None,
    truncation=True,
    device=DEVICE,
)

def predict_goemotions(text):
    preds = go_classifier(text)[0]
    return preds

def goemotions_to_va(preds, mapping):
    total_score = 0
    weighted_valence = 0
    weighted_arousal = 0

    for p in preds:
        label = p['label']
        score = p['score']
        if label in mapping:
            weighted_valence += mapping[label]['valence'] * score
            weighted_arousal += mapping[label]['arousal'] * score
            total_score += score

    if total_score == 0:
        return {'valence': None, 'arousal': None}

    return {
        'valence': weighted_valence / total_score,
        'arousal': weighted_arousal / total_score
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: SamLowe/roberta-base-go_emotions
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

RoBERTa

In [10]:
roberta_classifier = pipeline(
    task='text-classification',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    top_k=None,
    truncation=True,
    device=DEVICE,
)

def predict_roberta(text):
    return roberta_classifier(text)[0]

ROBERTA_VA = {
    'negative': {'valence': -0.8, 'arousal': 0.6},
    'neutral': {'valence': 0.0, 'arousal': 0.1},
    'positive': {'valence': 0.8, 'arousal': 0.6}
}

def roberta_to_va(preds, mapping):
    total_score = sum(p['score'] for p in preds)
    if total_score == 0:
        return {'valence': None, 'arousal': None}

    valence = sum(mapping[p['label']]['valence'] * p['score'] for p in preds if p['label'] in mapping)
    arousal = sum(mapping[p['label']]['arousal'] * p['score'] for p in preds if p['label'] in mapping)

    return {
        'valence': valence / total_score,
        'arousal': arousal / total_score
    }

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

VAD model

In [7]:
vad_classifier = pipeline(
    task='text-classification',
    model='RobroKools/vad-bert',
    tokenizer='RobroKools/vad-bert',
    top_k=None,
    truncation=True,
    max_length=512,
    device=DEVICE,
)

def predict_vad(text):
    preds = vad_classifier(text)[0]
    return preds

def _normalize_emobank(value):
    # EmoBank scale [1, 5] -> [-1, 1]: (x - 3) / 2
    return (value - 3) / 2

def vad_to_va(preds):
    # Model outputs LABEL_0=valence, LABEL_1=arousal, LABEL_2=dominance (regression values)
    scores = {p['label']: p['score'] for p in preds}
    return {
        'valence': _normalize_emobank(scores['LABEL_0']),
        'arousal': _normalize_emobank(scores['LABEL_1']),
        'dominance': _normalize_emobank(scores['LABEL_2']),
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/864 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: RobroKools/vad-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Sentence loop

In [8]:
def analyze_song_sentences(song_row, model_name, predictor, converter=None):
    title = song_row['title']
    artist = song_row['artist']
    lyrics = preprocess_lyrics(song_row['lyric'])
    sentences = split_into_sentences(lyrics)

    results = []

    for i, sentence in enumerate(sentences):
        try:
            raw_preds = predictor(sentence)

            if converter is not None:
                va = converter(raw_preds)
            else:
                va = raw_preds

            # go_top_emotion = None
            # roberta_class = None

            # if model_name == 'GoEmotions' and raw_preds:
            #     go_top_emotion = max(raw_preds, key=lambda p: p['score'])['label']

            # if model_name == 'RoBERTa' and raw_preds:
            #     roberta_class = max(raw_preds, key=lambda p: p['score'])['label']

            results.append({
                'title': title,
                'artist': artist,
                'model': model_name,
                'sentence_index': i,
                'sentence': sentence,
                'raw_output': str(raw_preds),
                # 'go_top_emotion': go_top_emotion,
                # 'roberta_class': roberta_class,
                'valence': va.get('valence'),
                'arousal': va.get('arousal'),
                'dominance': va.get('dominance', None)
            })

        except Exception as e:
            results.append({
                'title': title,
                'artist': artist,
                'model': model_name,
                'sentence_index': i,
                'sentence': sentence,
                'raw_output': f"Error: {str(e)}",
                # 'go_top_emotion': None,
                # 'roberta_class': None,
                'valence': None,
                'arousal': None,
                'dominance': None
            })

    return results

In [9]:
def aggregate_song_level(df):
    return (
        df.groupby(['artist', 'title', 'model'], as_index=False)
          .agg(
              mean_valence=('valence', 'mean'),
              mean_arousal=('arousal', 'mean'),
              mean_dominance=('dominance', 'mean')
          )
    )

Small test

In [10]:
TEST_SENTENCE = "I hurt myself today to see if I still feel, I focus on the pain, the only thing that's real"

tests = [
    # ("GoEmotions", lambda s: goemotions_to_va(predict_goemotions(s), GO_EMOTIONS_VA)),
    # ("RoBERTa",    lambda s: roberta_to_va(predict_roberta(s), ROBERTA_VA)),
    ("VAD",        lambda s: vad_to_va(predict_vad(s))),
]

print(f"Sentence: \"{TEST_SENTENCE}\"\n")
print(f"{'Model':<14} {'Valence':>10} {'Arousal':>10} {'Dominance':>12}")
print("-" * 50)
for name, fn in tests:
    result = fn(TEST_SENTENCE)
    v = f"{result.get('valence'):.4f}" if result.get('valence') is not None else "N/A"
    a = f"{result.get('arousal'):.4f}" if result.get('arousal') is not None else "N/A"
    d = f"{result.get('dominance'):.4f}" if result.get('dominance') is not None else "N/A"
    print(f"{name:<14} {v:>10} {a:>10} {d:>12}")

Sentence: "I hurt myself today to see if I still feel, I focus on the pain, the only thing that's real"

Model             Valence    Arousal    Dominance
--------------------------------------------------
VAD               -0.3905     0.2273      -0.0198


Bigger test on the dataset

In [ ]:
songs_df = analysis_df[['artist', 'title', 'disorder', 'lyric']].drop_duplicates(subset=['artist', 'title']).reset_index(drop=True)

sample_songs = songs_df.head(5)

model_runs = [
    # ('GoEmotions', predict_goemotions, lambda p: goemotions_to_va(p, GO_EMOTIONS_VA)),
    # ('RoBERTa',    predict_roberta,    lambda p: roberta_to_va(p, ROBERTA_VA)),
    ('VAD',        predict_vad,        vad_to_va),
]

all_results = []
for _, row in sample_songs.iterrows():
    for model_name, predictor, converter in model_runs:
        all_results.extend(analyze_song_sentences(row, model_name, predictor, converter))

sentence_df = pd.DataFrame(all_results)
agg_df = aggregate_song_level(sentence_df)

print(f"=== Sentence-level ({len(sentence_df)} rows) ===")
display(sentence_df[['artist', 'title', 'model', 'sentence_index', 'sentence', 'valence', 'arousal', 'dominance']])

print(f"\n=== Song-level aggregated ({len(agg_df)} rows) ===")
display(agg_df)

=== Sentence-level (174 rows) ===


,artist,title,model,sentence_index,sentence,go_top_emotion,roberta_class,valence,arousal,dominance
0,Quadeca,Burnin Bridges / Long Day (feat. IDK),GoEmotions,0,Highest To Lowest: Quadeca LyricsQuadeca's Son...,neutral,NaN,0.020092,0.020794,NaN
1,Quadeca,Burnin Bridges / Long Day (feat. IDK),GoEmotions,1,"(20,121,625)\n- BEAMIN' (8,999,725)\n- War!",neutral,NaN,0.011019,0.028546,NaN
2,Quadeca,Burnin Bridges / Long Day (feat. IDK),GoEmotions,2,(feat.,neutral,NaN,0.013172,0.018704,NaN
3,Quadeca,Burnin Bridges / Long Day (feat. IDK),GoEmotions,3,"Dax) (8,850,668)\n- Wii Music Fire (7,069,076)...",neutral,NaN,-0.143789,0.089599,NaN
4,Quadeca,Burnin Bridges / Long Day (feat. IDK),GoEmotions,4,"B. Lou) (6,183,298)\n- WHERE'D YOU GO?",neutral,NaN,0.044548,0.118747,NaN
...,...,...,...,...,...,...,...,...,...,...
169,Alice Cooper,School's Out,RoBERTa,0,"School’s Out Lyrics[Verse 1]\nWell, we got no ...",NaN,negative,-0.351280,0.375278,NaN
170,Alice Cooper,School's Out,VAD,0,"School’s Out Lyrics[Verse 1]\nWell, we got no ...",NaN,NaN,-0.165450,0.199404,0.061629
171,Shawn Mendes,Call My Friends,GoEmotions,0,"Call My Friends Lyrics[Verse 1]\nRight now, I'...",desire,NaN,-0.072849,0.091643,NaN
172,Shawn Mendes,Call My Friends,RoBERTa,0,"Call My Friends Lyrics[Verse 1]\nRight now, I'...",NaN,neutral,0.135476,0.274152,NaN


=== Sentence-level (174 rows) ===


,artist,title,model,sentence_index,sentence,go_top_emotion,roberta_class,valence,arousal,dominance
0,Quadeca,Burnin Bridges / Long Day (feat. IDK),GoEmotions,0,Highest To Lowest: Quadeca LyricsQuadeca's Son...,neutral,NaN,0.020092,0.020794,NaN
1,Quadeca,Burnin Bridges / Long Day (feat. IDK),GoEmotions,1,"(20,121,625)\n- BEAMIN' (8,999,725)\n- War!",neutral,NaN,0.011019,0.028546,NaN
2,Quadeca,Burnin Bridges / Long Day (feat. IDK),GoEmotions,2,(feat.,neutral,NaN,0.013172,0.018704,NaN
3,Quadeca,Burnin Bridges / Long Day (feat. IDK),GoEmotions,3,"Dax) (8,850,668)\n- Wii Music Fire (7,069,076)...",neutral,NaN,-0.143789,0.089599,NaN
4,Quadeca,Burnin Bridges / Long Day (feat. IDK),GoEmotions,4,"B. Lou) (6,183,298)\n- WHERE'D YOU GO?",neutral,NaN,0.044548,0.118747,NaN
...,...,...,...,...,...,...,...,...,...,...
169,Alice Cooper,School's Out,RoBERTa,0,"School’s Out Lyrics[Verse 1]\nWell, we got no ...",NaN,negative,-0.351280,0.375278,NaN
170,Alice Cooper,School's Out,VAD,0,"School’s Out Lyrics[Verse 1]\nWell, we got no ...",NaN,NaN,-0.165450,0.199404,0.061629
171,Shawn Mendes,Call My Friends,GoEmotions,0,"Call My Friends Lyrics[Verse 1]\nRight now, I'...",desire,NaN,-0.072849,0.091643,NaN
172,Shawn Mendes,Call My Friends,RoBERTa,0,"Call My Friends Lyrics[Verse 1]\nRight now, I'...",NaN,neutral,0.135476,0.274152,NaN



=== Song-level aggregated (15 rows) ===


,artist,title,model,mean_valence,mean_arousal,mean_dominance
0,Alice Cooper,School's Out,GoEmotions,-0.177140,0.024029,NaN
1,Alice Cooper,School's Out,RoBERTa,-0.351280,0.375278,NaN
2,Alice Cooper,School's Out,VAD,-0.165450,0.199404,0.061629
3,David Byrne,Lilies of the Valley,GoEmotions,0.038981,0.069575,NaN
4,David Byrne,Lilies of the Valley,RoBERTa,-0.343368,0.351536,NaN
5,David Byrne,Lilies of the Valley,VAD,-0.189674,0.225512,0.093785
6,Quadeca,Burnin Bridges / Long Day (feat. IDK),GoEmotions,0.006951,0.025027,NaN
7,Quadeca,Burnin Bridges / Long Day (feat. IDK),RoBERTa,0.053349,0.170344,NaN
8,Quadeca,Burnin Bridges / Long Day (feat. IDK),VAD,0.055654,0.060463,0.075335
9,Shawn Mendes,Call My Friends,GoEmotions,-0.072849,0.091643,NaN


In [44]:
# look at shool's out lyrics
school_out_lyrics = songs_df[songs_df['title'].str.lower() == "school's out"]['lyric'].values
school_out_lyrics

<StringArray>
['School’s Out Lyrics[Verse 1]\nWell, we got no choice\nAll the girls and boys\nMakin' all that noise\n'Cause they found new toys\nWell, we can't salute ya\nCan't find a flag\nIf that don't suit ya\nThat's a drag\n\n[Chorus]\nSchool's out for summer\nSchool's out forever\nSchool's been blown to pieces\n[Post-Chorus]\nNo more pencils\nNo more books\nNo more teacher's dirty looks, yeah\n\n[Verse 2]\nWell, we got no class\nAnd we got no principles\nAnd we got no innocence\nWe can't even think of a word that rhymes\n\n[Chorus]\nSchool's out for summer\nSchool's out forever\nMy school's been blown to pieces\n\n[Post-Chorus]\nNo more pencils\nNo more books\nNo more teacher's dirty looks\nOut for summer\nOut 'til fall\nWe might not come back at all\n\n[Outro]\nSchool's out forever\nSchool's out for summer\nSchool's out with fever\nSchool's out completely8Embed']
Length: 1, dtype: str

Final cell to extract the sentence sentiment per song

In [11]:
# Full run on selected songs (depression/control only) and save outputs
songs_df_full = (
    analysis_df[['artist', 'title', 'disorder', 'lyric']]
    .dropna(subset=['lyric'])
    .drop_duplicates(subset=['artist', 'title'])
    .reset_index(drop=True)
)

# Keep only target groups to speed up execution
target_groups = {'depression', 'control'}
songs_df_full = songs_df_full[
    songs_df_full['disorder'].astype(str).str.strip().str.lower().isin(target_groups)
].reset_index(drop=True)

# Build a base sentence table once
sentence_rows = []
for _, row in tqdm(songs_df_full.iterrows(), total=len(songs_df_full), desc='Preparing sentences'):
    lyrics = preprocess_lyrics(row['lyric'])
    for i, sentence in enumerate(split_into_sentences(lyrics)):
        sentence_rows.append({
            'artist': row['artist'],
            'title': row['title'],
            'disorder': row['disorder'],
            'sentence_index': i,
            'sentence': sentence,
        })

base_sentence_df = pd.DataFrame(sentence_rows)


def run_pipeline_with_cpu_fallback(clf, texts, batch_size, model_name):
    try:
        return clf(texts, batch_size=batch_size, truncation=True)
    except Exception as e:
        msg = str(e)
        if 'CUDA error' in msg or 'device-side assert' in msg:
            print(f"{model_name}: CUDA failure detected, retrying on CPU...")
            # if model_name == 'GoEmotions':
            #     cpu_clf = pipeline(
            #         task='text-classification',
            #         model='SamLowe/roberta-base-go_emotions',
            #         top_k=None,
            #         truncation=True,
            #         device=-1,
            #     )
            # elif model_name == 'RoBERTa':
            #     cpu_clf = pipeline(
            #         task='text-classification',
            #         model='cardiffnlp/twitter-roberta-base-sentiment-latest',
            #         top_k=None,
            #         truncation=True,
            #         device=-1,
            #     )
            # else:
            cpu_clf = pipeline(
                task='text-classification',
                model='RobroKools/vad-bert',
                tokenizer='RobroKools/vad-bert',
                top_k=None,
                truncation=True,
                max_length=512,
                device=-1,
            )
            return cpu_clf(texts, batch_size=batch_size, truncation=True)
        raise


if base_sentence_df.empty:
    sentence_df_full = pd.DataFrame(columns=[
        'artist', 'title', 'disorder', 'sentence_index', 'sentence', 'model',
        'raw_output', 'valence', 'arousal', 'dominance'
    ])
    song_level_df_full = pd.DataFrame(columns=[
        'artist', 'title', 'model', 'mean_valence', 'mean_arousal', 'mean_dominance'
    ])
    song_level_wide = pd.DataFrame(columns=['artist', 'title'])
else:
    texts = base_sentence_df['sentence'].tolist()

    # # GoEmotions (batched)
    # go_outputs = run_pipeline_with_cpu_fallback(go_classifier, texts, BATCH_SIZE_GO, 'GoEmotions')
    # go_records = []
    # for out in tqdm(go_outputs, total=len(go_outputs), desc='Processing GoEmotions'):
    #     top_emotion = max(out, key=lambda p: p['score'])['label'] if out else None
    #     va = goemotions_to_va(out, GO_EMOTIONS_VA)
    #     go_records.append({
    #         'model': 'GoEmotions',
    #         'raw_output': str(out),
    #         'go_top_emotion': top_emotion,
    #         'roberta_class': None,
    #         'valence': va.get('valence'),
    #         'arousal': va.get('arousal'),
    #         'dominance': None,
    #     })
    # go_df = pd.concat([base_sentence_df.reset_index(drop=True), pd.DataFrame(go_records)], axis=1)

    # # RoBERTa (batched)
    # roberta_outputs = run_pipeline_with_cpu_fallback(roberta_classifier, texts, BATCH_SIZE_ROBERTA, 'RoBERTa')
    # roberta_records = []
    # for out in tqdm(roberta_outputs, total=len(roberta_outputs), desc='Processing RoBERTa'):
    #     top_class = max(out, key=lambda p: p['score'])['label'] if out else None
    #     va = roberta_to_va(out, ROBERTA_VA)
    #     roberta_records.append({
    #         'model': 'RoBERTa',
    #         'raw_output': str(out),
    #         'go_top_emotion': None,
    #         'roberta_class': top_class,
    #         'valence': va.get('valence'),
    #         'arousal': va.get('arousal'),
    #         'dominance': None,
    #     })
    # roberta_df = pd.concat([base_sentence_df.reset_index(drop=True), pd.DataFrame(roberta_records)], axis=1)

    # VAD (batched)
    vad_outputs = run_pipeline_with_cpu_fallback(vad_classifier, texts, BATCH_SIZE_VAD, 'VAD')
    vad_records = []
    for out in tqdm(vad_outputs, total=len(vad_outputs), desc='Processing VAD'):
        va = vad_to_va(out)
        vad_records.append({
            'model': 'VAD',
            'raw_output': str(out),
            # 'go_top_emotion': None,
            # 'roberta_class': None,
            'valence': va.get('valence'),
            'arousal': va.get('arousal'),
            'dominance': va.get('dominance'),
        })
    vad_df = pd.concat([base_sentence_df.reset_index(drop=True), pd.DataFrame(vad_records)], axis=1)

    # sentence_df_full = pd.concat([go_df, roberta_df, vad_df], ignore_index=True)
    sentence_df_full = vad_df
    song_level_df_full = aggregate_song_level(sentence_df_full)

    # Optional wide format for quick comparison across models
    song_level_wide = song_level_df_full.pivot_table(
        index=['artist', 'title'],
        columns='model',
        values=['mean_valence', 'mean_arousal', 'mean_dominance'],
    )
    song_level_wide.columns = [f"{metric}_{model}" for metric, model in song_level_wide.columns]
    song_level_wide = song_level_wide.reset_index()

# Save outputs (Drive if mounted, otherwise current directory)
output_dir = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').exists() else Path('.')
output_dir.mkdir(parents=True, exist_ok=True)

sentence_path = output_dir / 'sentiment_sentence_level_enriched.csv'
song_path = output_dir / 'sentiment_song_level_enriched.csv'
song_wide_path = output_dir / 'sentiment_song_level_wide_enriched.csv'

sentence_df_full.to_csv(sentence_path, index=False)
song_level_df_full.to_csv(song_path, index=False)
song_level_wide.to_csv(song_wide_path, index=False)

print(f"Groups included: {sorted(target_groups)}")
print(f"Songs processed: {len(songs_df_full)}")
print(f"Unique sentences processed: {len(base_sentence_df)}")
print(f"Sentence-level rows (VAD): {len(sentence_df_full)}")
print(f"Saved sentence-level file: {sentence_path}")
print(f"Saved song-level file: {song_path}")
print(f"Saved song-level wide file: {song_wide_path}")

display(sentence_df_full.head())
display(song_level_df_full.head())

Preparing sentences:   0%|          | 0/35571 [00:00<?, ?it/s]

Processing VAD:   0%|          | 0/2779562 [00:00<?, ?it/s]

Groups included: ['control', 'depression']
Songs processed: 35571
Unique sentences processed: 2779562
Sentence-level rows (VAD): 2779562
Saved sentence-level file: /content/drive/MyDrive/sentiment_sentence_level_enriched.csv
Saved song-level file: /content/drive/MyDrive/sentiment_song_level_enriched.csv
Saved song-level wide file: /content/drive/MyDrive/sentiment_song_level_wide_enriched.csv


,artist,title,disorder,sentence_index,sentence,model,raw_output,valence,arousal,dominance
0,Quadeca,Burnin Bridges / Long Day (feat. IDK),depression,0,Highest To Lowest: Quadeca LyricsQuadeca's Son...,VAD,"[{'label': 'LABEL_1', 'score': 3.3642196655273...",0.143427,0.182110,0.049787
1,Quadeca,Burnin Bridges / Long Day (feat. IDK),depression,1,"(20,121,625)\n- BEAMIN' (8,999,725)\n- War!",VAD,"[{'label': 'LABEL_1', 'score': 3.6385209560394...",0.114133,0.319260,0.127583
2,Quadeca,Burnin Bridges / Long Day (feat. IDK),depression,2,(feat.,VAD,"[{'label': 'LABEL_2', 'score': 3.1772220134735...",0.065396,0.059895,0.088611
3,Quadeca,Burnin Bridges / Long Day (feat. IDK),depression,3,"Dax) (8,850,668)\n- Wii Music Fire (7,069,076)...",VAD,"[{'label': 'LABEL_1', 'score': 3.1627085208892...",-0.001970,0.081354,0.067098
4,Quadeca,Burnin Bridges / Long Day (feat. IDK),depression,4,"B. Lou) (6,183,298)\n- WHERE'D YOU GO?",VAD,"[{'label': 'LABEL_2', 'score': 3.0688900947570...",0.015637,-0.030077,0.034445


,artist,title,model,mean_valence,mean_arousal,mean_dominance
0,"""Weird Al"" Yankovic",Fat,VAD,-0.033980,0.173124,0.100035
1,#Freekd,ACTUALLY,VAD,-0.073895,0.242156,0.115553
2,#Freekd,JUMPOFF,VAD,0.011588,0.159050,0.082422
3,$NOT,Moon & Stars (feat. Maggie Lindemann),VAD,0.054230,0.107910,0.069382
4,$UICIDEBOY$,FUCKTHEPOPULATION,VAD,-0.305391,0.253423,0.128955
